# Customer Churn Prediction System
## End-to-End Reproducible Machine Learning, Explainable AI, and Customer Risk Analysis
**Academic & Reproducibility Notice**:
The baseline dataset used in this notebook is synthetically generated (=1000$, 41.4% churn rate) to demonstrate the machine learning pipeline, feature engineering, and explainable AI architecture.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib

print("Libraries loaded successfully!")


In [2]:
# Synthetic Dataset Generation (Exact logic from baseline)
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    "CustomerID": [f"CUST-{i:04d}" for i in range(1, n_samples + 1)],
    "Tenure_Months": np.random.randint(1, 72, size=n_samples),
    "MonthlyCharges": np.round(np.random.uniform(20.0, 120.0, size=n_samples), 2),
    "ContractType": np.random.choice(["Month-to-month", "One year", "Two year"], size=n_samples, p=[0.5, 0.3, 0.2]),
    "InternetService": np.random.choice(["DSL", "Fiber optic", "No"], size=n_samples, p=[0.4, 0.4, 0.2]),
    "PaperlessBilling": np.random.choice(["Yes", "No"], size=n_samples, p=[0.6, 0.4]),
    "PaymentMethod": np.random.choice(["Electronic check", "Mailed check", "Bank transfer", "Credit card"], size=n_samples),
})

data["TotalCharges"] = np.round(data["Tenure_Months"] * data["MonthlyCharges"] + np.random.normal(0, 10, size=n_samples), 2)
data["TotalCharges"] = np.maximum(data["TotalCharges"], 0.0)

churn_prob = (
    (72 - data["Tenure_Months"]) / 72 * 0.4 +
    (data["ContractType"] == "Month-to-month") * 0.3 +
    (data["MonthlyCharges"] / 120.0) * 0.3
)
churn_prob = (churn_prob - churn_prob.min()) / (churn_prob.max() - churn_prob.min())
data["Churn"] = np.where(churn_prob > 0.55, 1, 0)

print(f"Dataset Shape: {data.shape}")
print(f"Class Distribution:
{data['Churn'].value_counts(normalize=True)}")


In [3]:
# Feature Engineering
data["MonthlyToTotalRatio"] = data["MonthlyCharges"] / (data["TotalCharges"] + 1.0)
data["IsNewCustomer"] = (data["Tenure_Months"] <= 6).astype(int)

# Unbiased feature set: drop CustomerID, Churn, and ContractType
X_unbiased = data.drop(columns=["CustomerID", "Churn", "ContractType"])
y = data["Churn"]

# Train/Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_unbiased, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


In [4]:
# Preprocessing Pipeline (Fit on train only to prevent leakage)
num_cols = ["Tenure_Months", "MonthlyCharges", "TotalCharges", "MonthlyToTotalRatio", "IsNewCustomer"]
cat_cols = ["InternetService", "PaperlessBilling", "PaymentMethod"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
    ]
)

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, y_proba):.4f}")
print("
Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix (TN, FP / FN, TP):")
print(confusion_matrix(y_test, y_pred))


In [5]:
# Single Customer Prediction (Cell 8 verification)
test_customer = pd.DataFrame([{
    "Tenure_Months": 24,
    "MonthlyCharges": 85.50,
    "TotalCharges": 256.50,
    "InternetService": "Fiber optic",
    "PaperlessBilling": "No",
    "PaymentMethod": "Electronic check"
}])
test_customer["MonthlyToTotalRatio"] = test_customer["MonthlyCharges"] / (test_customer["TotalCharges"] + 1.0)
test_customer["IsNewCustomer"] = (test_customer["Tenure_Months"] <= 6).astype(int)

pred_prob = baseline_pipeline.predict_proba(test_customer)[0, 1]
print(f"Sample Customer Churn Probability: {pred_prob * 100:.2f}%")
print(f"Status: {'CHURN RISK (High Risk)' if pred_prob >= 0.60 else 'Likely to Stay'}")
